In [1]:
import csv
import json
import re

from scielo_scholarly_data import standardizer
from scielo_scholarly_data.standardizer import ImpossibleConvertionToIntError, InvalidRomanNumeralError

In [2]:
path_sci_source = '/home/rafaeljpd/Downloads/SCI_RegsC_TitCorr_TB.txt'
path_sci_output = '/home/rafaeljpd/Downloads/output.sci.json'
path_sci_output_missing = '/home/rafaeljpd/Downloads/output.sci.missing.json'

sci_source_fieldnames = [
    'id',
    'citation_count',
    'cited_doiset',
    'cited_journal',
    'cited_year',
    'cited_vol',
]

regex_volume = r'^(?P<posfix>[v|V])(?P<volume>\d*)$'

In [3]:
def fix_volume(text):
	if text.isdigit():
		return text

	m = re.match(regex_volume, text)
	if m:
		return m.groupdict().get('volume')

	try:
		return standardizer.issue_volume(text)
	except (ImpossibleConvertionToIntError, InvalidRomanNumeralError):
		return standardizer.issue_volume(text, force_integer=False)


def standardize_data(data):        
    if 'cited_doiset' in data or 'cited_doi' in data:
        cited_doiset = set()
        if data['cited_doiset'] is None:
            print(data)
        else:
            for d in data['cited_doiset'].split(' '):
                doi_stz = standardizer.document_doi(d, return_mode='path')
                if not isinstance(doi_stz, dict):
                    cited_doiset.add(doi_stz)
        if len(cited_doiset) > 0:
            data['cited_doiset'] = '#'.join(cited_doiset)

    if 'cited_vol' in data:
        fixed_vol = fix_volume(data['cited_vol'])
        data['cited_vol'] = fixed_vol

    if 'cited_journal' in data:
          data['cited_journal'] = data['cited_journal'].strip()
        
def genkey(data):
	standardize_data(data)
        
	return '|'.join([
        data['id'],
        data['citation_count'],
        data['cited_doiset'],
        data['cited_journal'],
        data['cited_year'],
        data['cited_vol']
    ])

In [4]:
sci_output = []

with open(path_sci_output) as fin:
    for line in fin:
        jline = json.loads(line)
        sci_output.append(jline)

sci_output

[{'citation_count': '6323',
  'cited_doiset': '',
  'cited_issnl': '1932-6203',
  'cited_journal': 'PLOS ONE',
  'cited_vol': '8',
  'cited_year': '2013',
  'id': '999993676',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '5713',
  'cited_doiset': '',
  'cited_issnl': '1932-6203',
  'cited_journal': 'PLOS ONE',
  'cited_vol': '9',
  'cited_year': '2014',
  'id': '999994286',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '5290',
  'cited_doiset': '',
  'cited_issnl': '1932-6203',
  'cited_journal': 'PLOS ONE',
  'cited_vol': '7',
  'cited_year': '2012',
  'id': '999994709',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '5251',
  'cited_doiset': '',
  'cited_issnl': '1932-6203',
  'cited_journal': 'PLOS ONE',
  'cited_vol': '10',
  'cited_year': '2015',
  'id': '999994748',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '4497',
  'cited_doiset': '',
  'cited_issnl': '0102-311X',
  'cited_journal': 'CAD SAUDE PUBLICA',
  'cited_vo

In [5]:
sci_output_missing = []

with open(path_sci_output_missing) as fin:
    for line in fin:
        jline = json.loads(line)
        sci_output_missing.append(jline)

sci_output_missing

[{'citation_count': '2',
  'cited_doiset': '10.1007/BF01193981#10.1007/BF01201643',
  'cited_issnl': '0168-8162',
  'cited_journal': 'EXPERIMENTAL AND APPLIED ACAROLOGY',
  'cited_vol': '6',
  'cited_year': '1989',
  'id': '999999997',
  'issnls_size': 1,
  'result_code': 0},
 {'citation_count': '1',
  'cited_doiset': '10.1001/JAMA.2018.9071.#10.1001/JAMA.2018.9071',
  'cited_issnl': '0098-7484',
  'cited_journal': 'JAMA',
  'cited_vol': '320',
  'cited_year': '2018',
  'id': '999999998',
  'issnls_size': 20,
  'result_code': 1,
  'title_year_volume_key': 'JAMA-2018-320'},
 {'citation_count': '1',
  'cited_doiset': '10.1001/JAMA.2020.4756.#10.1001/JAM',
  'cited_journal': 'JAMA',
  'cited_vol': '',
  'cited_year': '2020',
  'id': '999999998',
  'issnls_size': 20,
  'result_code': 529},
 {'citation_count': '1',
  'cited_doiset': '10.1016/S0140-6736(08)60459-6.#10.1016/S0140-',
  'cited_issnl': '0140-6736',
  'cited_journal': 'LANCET',
  'cited_vol': '371',
  'cited_year': '2008',
  'id'

In [6]:
sci_source = []

with open(path_sci_source, errors='ignore') as fin:
    cr = csv.DictReader(fin, fieldnames=sci_source_fieldnames, delimiter='|', restval='', escapechar='\\', quoting=csv.QUOTE_NONE)
    for row in cr:
        standardize_data(row)
        sci_source.append(row)

sci_source

[{'id': '999993676',
  'citation_count': '6323',
  'cited_doiset': '',
  'cited_journal': 'PLOS ONE',
  'cited_year': '2013',
  'cited_vol': '8'},
 {'id': '999994286',
  'citation_count': '5713',
  'cited_doiset': '',
  'cited_journal': 'PLOS ONE',
  'cited_year': '2014',
  'cited_vol': '9'},
 {'id': '999994709',
  'citation_count': '5290',
  'cited_doiset': '',
  'cited_journal': 'PLOS ONE',
  'cited_year': '2012',
  'cited_vol': '7'},
 {'id': '999994748',
  'citation_count': '5251',
  'cited_doiset': '',
  'cited_journal': 'PLOS ONE',
  'cited_year': '2015',
  'cited_vol': '10'},
 {'id': '999995502',
  'citation_count': '4497',
  'cited_doiset': '',
  'cited_journal': 'CAD SAUDE PUBLICA',
  'cited_year': '2008',
  'cited_vol': '24'},
 {'id': '999995876',
  'citation_count': '4123',
  'cited_doiset': '',
  'cited_journal': 'CAD SAUDE PUBLICA',
  'cited_year': '2003',
  'cited_vol': '19'},
 {'id': '999996042',
  'citation_count': '3957',
  'cited_doiset': '',
  'cited_journal': 'CAD SA

In [7]:
len(sci_source), len(sci_output), len(sci_output_missing)

(5212182, 4901343, 310924)

In [8]:
sci_source[2974958]

{'id': '999999998',
 'citation_count': '1',
 'cited_doiset': '10.1002/JDD.12470',
 'cited_journal': 'JOURNAL OF DENTAL EDUCATION',
 'cited_year': '2021',
 'cited_vol': '85'}

In [9]:
sci_output[2974958]

{'citation_count': '1',
 'cited_doiset': '10.1016/J.PDPDT.2019.01.020',
 'cited_issnl': '1572-1000',
 'cited_journal': 'PHOTODIAGN PHOTODYN THER',
 'cited_vol': '25',
 'cited_year': '2019',
 'id': '999999998',
 'issnls_size': 1,
 'result_code': 11,
 'title_year_volume_key': 'PHOTODIAGNOSIS AND PHOTODYNAMIC THERAPY-2019-25'}

In [10]:
sci_output_missing[0]

{'citation_count': '2',
 'cited_doiset': '10.1007/BF01193981#10.1007/BF01201643',
 'cited_issnl': '0168-8162',
 'cited_journal': 'EXPERIMENTAL AND APPLIED ACAROLOGY',
 'cited_vol': '6',
 'cited_year': '1989',
 'id': '999999997',
 'issnls_size': 1,
 'result_code': 0}

In [11]:
source_key_to_line_number = {}
for ind, s in enumerate(sci_source):
    k = genkey(s)
    if k not in source_key_to_line_number:
        source_key_to_line_number[k] = []
    source_key_to_line_number[k].append(ind)

In [12]:
output_key_to_line_number = {}
for ind, o in enumerate(sci_output):
    k = genkey(o)
    if k not in output_key_to_line_number:
        output_key_to_line_number[k] = []
    output_key_to_line_number[k].append(ind)

In [13]:
output_missing_key_to_line_number = {}
for ind, om in enumerate(sci_output_missing):
    k = genkey(om)
    if k not in output_missing_key_to_line_number:
        output_missing_key_to_line_number[k] = []
    output_missing_key_to_line_number[k].append(ind)

In [14]:
for k, v in source_key_to_line_number.items():
    if len(v) > 2:
        print(k, v)
        break

999999995|4||PHYS REV|1985|31 [567626, 567627, 567628, 567629]


In [15]:
for k, v in output_key_to_line_number.items():
    if len(v) > 2:
        print(k, v)
        break

999999995|4||PHYS REV|1985|31 [567626, 567627, 567628, 567629]


In [16]:
for k, v in output_missing_key_to_line_number.items():
    if len(v) > 2:
        print(k, v)
        break

999999998|1||ANALES DE ARQUEOLOGIA Y ETNOLOGIA|1958|14 [2560, 2561, 2562]


In [17]:
k1 = '999999998|1| 3, 2017, 73-99, DISPONIBLE EN: HTTP://ONLINELIBRARY.WILEY.COM/D|THE JOURNAL OF PATHOLOGY CLINICAL RESEARCH|2017|'

for l in source_key_to_line_number[k1]:
    print(sci_source[l])

for k in output_key_to_line_number:
    if '|THE JOURNAL OF PATHOLOGY CLINICAL RESEARCH' in k:
        print(k)

{'id': '999999998', 'citation_count': '1', 'cited_doiset': ' 3, 2017, 73-99, DISPONIBLE EN: HTTP://ONLINELIBRARY.WILEY.COM/D', 'cited_journal': 'THE JOURNAL OF PATHOLOGY CLINICAL RESEARCH', 'cited_year': '2017', 'cited_vol': ''}
999999998|1|3, 2017, 73-99, DISPONIBLE EN: HTTP://ONLINELIBRARY.WILEY.COM/D|THE JOURNAL OF PATHOLOGY CLINICAL RESEARCH|2017|
999999998|1||THE JOURNAL OF PATHOLOGY CLINICAL RESEARCH|2017|3


In [18]:
for k in output_key_to_line_number:
    if '999999998|1||ANNALES DE LINSTITUT PASTEUR|1993|4' in k:
        for l in output_key_to_line_number[k]:
            print(sci_output[l])

In [19]:
for k in output_missing_key_to_line_number:
    if '999999998|1||ANNALES DE LINSTITUT PASTEUR|1993|4' in k:
        for l in output_missing_key_to_line_number[k]:
            print(sci_output_missing[l])

{'citation_count': '1', 'cited_doiset': '', 'cited_journal': 'ANNALES DE LINSTITUT PASTEUR', 'cited_vol': '4', 'cited_year': '1993', 'id': '999999998', 'result_code': 70}
{'citation_count': '1', 'cited_doiset': '', 'cited_journal': 'ANNALES DE LINSTITUT PASTEUR', 'cited_vol': '4', 'cited_year': '1993', 'id': '999999998', 'result_code': 70}
{'citation_count': '1', 'cited_doiset': '', 'cited_journal': 'ANNALES DE LINSTITUT PASTEUR', 'cited_vol': '4', 'cited_year': '1993', 'id': '999999998', 'result_code': 70}


In [24]:
def extract_value(data):
    return '|'.join([
        data.get('cited_issnl', ''),
        str(data.get('result_code', '')),
        str(data.get('issnls_size', '')),
        data.get('title_year_volume_key', '')
    ]), len(data.get('cited_issnl', '')) == 9

def find_best_result(chave, output_keys_1, output_keys_2):
    o1_lines = output_keys_1.get(chave, [])
    o2_lines = output_keys_2.get(chave, [])

    outs_with_issnl = set()
    outs = set()

    for ol in o1_lines:
        ol_v, has_issnl = extract_value(sci_output[ol])
        outs.add(ol_v)
        if has_issnl:
            outs_with_issnl.add(ol_v)

    for ol in o2_lines:
        ol_v, has_issnl = extract_value(sci_output_missing[ol])
        outs.add(ol_v)
        if has_issnl:
            outs_with_issnl.add(ol_v)

    # espera-se que todos os ISSNLs atribuídos para uma chave sejam os mesmos
    # então, pode-se retornar o primeiro
    if outs_with_issnl:
        return outs_with_issnl.pop()

    # espera-se que todos os erros identificados para uma chave sejam parecidas
    # então, pode-se retornar o primeiro
    try:
        return outs.pop()
    except:
        print(chave)
        return ''

source_enriched = {}

null_value = '|'.join(['', '', '', ''])

for k in source_key_to_line_number:
    v = find_best_result(k, output_key_to_line_number, output_missing_key_to_line_number)

    lines = source_key_to_line_number[k]    
    for l in lines:
        if len(v) > 0:
            source_enriched[l] = f'{k}|{v}'
        else:
            source_enriched[l] = f'{k}|{null_value}'

999999998|1|10.1186/1465-9921-7-44.|RESPIR RES|2006|7
999999998|1| 3, 2017, 73-99, DISPONIBLE EN: HTTP://ONLINELIBRARY.WILEY.COM/D|THE JOURNAL OF PATHOLOGY CLINICAL RESEARCH|2017|
999999998|1|10.1111/J.1523-|CONSERV BIOL|1998|12
999999998|1|10.1177/1087054711427530.|JOURNAL OF ATTENTION DISORDERS|2012|16
999999998|1|10.2307/2641410|ECOLOGICAL APPLICATIONS|1999|9
999999998|1|10.32811/2595|REV SAUDE PUBLICA PARANA|2018|1
999999998|1|10.1080/030|THE VOCATIONAL ASPECT OF EDUCATION|2006|47
999999998|1|10.1172/JCI2|J CLIN INVEST|2003|112
999999998|1|10.2307/255|AFRICAN JOURNAL OF REPRODUCTIVE HEALTH|2007|11


In [25]:
with open('/home/rafaeljpd/Downloads/output.sci.fixed.csv', 'w') as fout:
    for k in sorted(source_enriched):
        fout.write(source_enriched[k] + '\n')

In [27]:
def to_json(data):
    return json.dumps(data, default=lambda o: o.__dict__, sort_keys=True, ensure_ascii=False)

def load_citation_from_str(str_data):
    data = {}

    els = str_data.split('|')

    if len(els) != 10:
        print(str_data)

    data['id'] = els[0].strip()
    data['citation_count'] = els[1].strip()
    data['cited_doiset'] = els[2].strip()
    data['cited_journal'] = els[3].strip()
    data['cited_year'] = els[4].strip()
    data['cited_vol'] = els[5].strip()
    
    if '<<<PROCESS>>>' not in str_data:
        data['cited_issnl'] = els[6].strip()
        try:
            data['result_code'] = int(els[7].strip())
        except ValueError:
            data['result_code'] = ''

        try:    
            data['issnls_size'] = int(els[8].strip())
        except ValueError:
            data['issnls_size'] = ''

        data['title_year_volume_key'] = els[9].strip()

    return data
    
with open('/home/rafaeljpd/Downloads/output.sci.fixed.json', 'w') as fout:
    for k in sorted(source_enriched):
        v = load_citation_from_str(source_enriched[k])
        fout.write(to_json(v) + '\n')